# Healthcare KG — Population Intelligence V1
**Isolated Google Colab pilot · 24 October 2026 baseline**

**Research question:** How many U.S. adults have laboratory-defined CKD and *probable* type 2 diabetes, by age, sex and race/ethnicity?

This notebook **does not modify GitHub**, does not require an OpenAI key, and never writes to the original KG. It downloads public **NHANES 2017–2018** files, constructs transparent *cross-sectional proxies*, computes weighted national estimates with design-based approximate standard errors, and exports CSVs plus an immutable baseline verification report.

**Scientific caveats:** One-time eGFR/UACR measurements cannot establish chronic kidney disease for ≥3 months; NHANES does not directly establish clinical CKD *attribution* to diabetes; self-reported diabetes diagnosed at age ≥30 is only a **proxy**, not a confirmed type 2 diabetes diagnosis. Therefore results are *not* labeled 'CKD associated with T2D' or Kerendia-eligible patients. This is a reproducible feasibility pilot, not a market-size estimate.

In [ ]:
%pip -q install pandas numpy scipy requests networkx pyreadstat

## 1. Separate storage and production baseline
Upload your October 24 ZIP when prompted. All outputs are saved to a new local `/content/healthcare_kg_population_v1/` directory. Optionally copy that directory to Google Drive **after** the run. No GitHub credentials or write access are used.

In [ ]:
from pathlib import Path
import zipfile, json, hashlib, shutil, datetime
from google.colab import files
ROOT=Path('/content/healthcare_kg_population_v1'); ROOT.mkdir(parents=True,exist_ok=True)
print('Upload 24OCT26_Healthcare_KG_Archive.zip (production snapshot; read-only input)')
uploaded=files.upload()
zipname=next((k for k in uploaded if k.lower().endswith('.zip')),None)
assert zipname, 'Please upload the October 24 ZIP archive.'
archive=Path('/content')/zipname
with zipfile.ZipFile(archive) as z:
    member=next(x for x in z.namelist() if x.startswith('medical_kg') and x.endswith('.json'))
    raw=z.read(member)
    kg=json.loads(raw)
(ROOT/'baseline_kg_COPY.json').write_bytes(raw)
base_hash=hashlib.sha256(raw).hexdigest()
node_count=len(kg['nodes']); edge_count=len(kg['edges'])
print('Baseline nodes:',node_count,'edges:',edge_count,'SHA256:',base_hash)
assert node_count>0 and edge_count>0

## 2. Download public NHANES datasets
CDC NHANES 2017–2018: demographics (`DEMO_J`), urine albumin/creatinine (`ALB_CR_J`), biochemistry (`BIOPRO_J`), and diabetes questionnaire (`DIQ_J`). Downloaded files are cached locally. If CDC is temporarily unavailable, rerun later; **no fabricated substitute records** are generated.

In [ ]:
import pandas as pd, numpy as np, requests, io, time
BASE='https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/'
SOURCES={name:BASE+name+'.xpt' for name in ['DEMO_J','ALB_CR_J','BIOPRO_J','DIQ_J']}
def get_xpt(name,url):
    path=ROOT/(name+'.xpt')
    if not path.exists():
        response=requests.get(url,timeout=90,headers={'User-Agent':'Healthcare-KG-Population-Research/1.0'})
        response.raise_for_status()
        if len(response.content)<1000: raise ValueError(f'Unexpected small file: {url}')
        path.write_bytes(response.content)
    return pd.read_sas(path,format='xport')
tables={name:get_xpt(name,url) for name,url in SOURCES.items()}
for name,frame in tables.items(): print(name,len(frame),'rows',len(frame.columns),'columns')

## 3. Define transparent clinical proxies
- **Adult:** age ≥20.
- **Laboratory CKD screen:** eGFR <60 mL/min/1.73m² **or** UACR ≥30 mg/g.
- **eGFR:** 2021 CKD-EPI creatinine equation (race-free); creatinine in mg/dL.
- **Diabetes proxy:** respondent reports diagnosed diabetes (`DIQ010 == 1`).
- **Probable T2D proxy:** diagnosed diabetes and self-reported age at diagnosis ≥30 (`DIQ040`). This excludes younger-onset T2D and may include adult-onset T1D. Report results as *proxy only*.
- **UACR:** urine albumin mg/L ×100 / urine creatinine mg/dL.

These data cannot establish persistence of CKD or causation by diabetes. Missing values are excluded *per estimate*, not converted to negative diagnoses.

In [ ]:
d=tables['DEMO_J'][['SEQN','RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU']].copy()
for name in ['ALB_CR_J','BIOPRO_J','DIQ_J']:
    d=d.merge(tables[name],on='SEQN',how='left',validate='one_to_one')
needed=['RIDAGEYR','RIAGENDR','RIDRETH3','WTMEC2YR','SDMVSTRA','SDMVPSU','URXUMA','URXUCR','LBXSCR','DIQ010','DIQ040']
missing=[x for x in needed if x not in d.columns]
assert not missing, f'NHANES variable names changed or unavailable: {missing}'
d=d[d.RIDAGEYR>=20].copy()
for col in needed: d[col]=pd.to_numeric(d[col],errors='coerce')
# 2021 CKD-EPI creatinine formula, Inker et al. NEJM 2021
female=d.RIAGENDR.eq(2)
kappa=np.where(female,0.7,0.9)
alpha=np.where(female,-0.241,-0.302)
scr=d.LBXSCR
ratio=scr/kappa
d['egfr']=142*np.minimum(ratio,1)**alpha*np.maximum(ratio,1)**(-1.200)*0.9938**d.RIDAGEYR*np.where(female,1.012,1.0)
d.loc[(scr<=0)|scr.isna(),'egfr']=np.nan
d['uacr']=100*d.URXUMA/d.URXUCR
d.loc[(d.URXUCR<=0)|d.URXUCR.isna(),'uacr']=np.nan
d['ckd_screen']=np.where(d.egfr.notna()&d.uacr.notna(),(d.egfr<60)|(d.uacr>=30),np.nan)
d['diagnosed_diabetes']=np.where(d.DIQ010.isin([1,2]),d.DIQ010.eq(1),np.nan)
d['probable_t2d_proxy']=np.where(d.diagnosed_diabetes.eq(1)&d.DIQ040.notna()&d.DIQ040.between(1,100),d.DIQ040>=30,np.where(d.diagnosed_diabetes.eq(0),False,np.nan))
d['joint_proxy']=np.where(d.ckd_screen.notna()&d.probable_t2d_proxy.notna(),d.ckd_screen.astype(bool)&d.probable_t2d_proxy.astype(bool),np.nan)
d['age_group']=pd.cut(d.RIDAGEYR,bins=[20,45,65,75,np.inf],right=False,labels=['20-44','45-64','65-74','75+'])
d['sex']=d.RIAGENDR.map({1:'Male',2:'Female'})
d['race_ethnicity']=d.RIDRETH3.map({1:'Mexican American',2:'Other Hispanic',3:'Non-Hispanic White',4:'Non-Hispanic Black',6:'Non-Hispanic Asian',7:'Other/multiracial'})
print('Adult rows:',len(d),'Joint proxy evaluable:',d.joint_proxy.notna().sum())

## 4. Complex survey weighted estimates
NHANES MEC exam weights (`WTMEC2YR`) represent the civilian noninstitutionalized U.S. population. Standard errors use **Taylor linearization with NHANES masked strata and PSUs**. Small cells and unstable estimates are flagged. Results are *2017–2018* estimates, **not current 2026 population counts**. Population total estimates for this two-year cycle use the 2-year MEC weights directly.

In [ ]:
from scipy.stats import t as student_t
# For weighted domain totals: sum(w*y). Variance uses between-PSU variance within each masked stratum.
def taylor_total_variance(frame, variable):
    f=frame.copy()
    f['_contrib']=f['WTMEC2YR']*f[variable].fillna(0).astype(float)
    psu=f.groupby(['SDMVSTRA','SDMVPSU'],observed=True)['_contrib'].sum()
    var=0.0; strata=0; psus=0
    for _,g in psu.groupby(level=0):
        vals=g.to_numpy(); m=len(vals)
        if m>1:
            var+=m/(m-1)*np.sum((vals-vals.mean())**2)
            strata+=1; psus+=m
    return float(var), max(1,psus-strata)
def estimate(frame,dimension,segment):
    # use all adult survey participants for domain estimation, setting outside-domain contributions to zero
    z=frame.copy()
    domain=(pd.Series(True,index=z.index) if dimension=='All adults' else z[dimension].eq(segment))
    valid=z.joint_proxy.notna() & z.WTMEC2YR.gt(0) & z.SDMVSTRA.notna() & z.SDMVPSU.notna()
    z['_case']=(domain&valid&z.joint_proxy.eq(1)).astype(float)
    z['_denom']=(domain&valid).astype(float)
    count=(z.WTMEC2YR*z._case).sum()
    denom=(z.WTMEC2YR*z._denom).sum()
    var_count,dfree=taylor_total_variance(z,'_case')
    # linearization for ratio prevalence
    z['_ratio_resid']=np.where(denom>0,z._case-count/denom*z._denom,0)
    var_ratio,_=taylor_total_variance(z,'_ratio_resid')
    se_count=np.sqrt(var_count); se_prev=np.sqrt(var_ratio)/denom if denom else np.nan
    critical=student_t.ppf(.975,dfree)
    prev=count/denom if denom else np.nan
    n_domain=int((domain&valid).sum()); n_cases=int((domain&valid&z.joint_proxy.eq(1)).sum())
    return {'dimension':dimension,'segment':str(segment),'unweighted_n':n_domain,'unweighted_cases':n_cases,
            'weighted_population_count':round(count),'count_ci_low':max(0,round(count-critical*se_count)),
            'count_ci_high':round(count+critical*se_count),'prevalence':prev,
            'prevalence_ci_low':max(0,prev-critical*se_prev),'prevalence_ci_high':min(1,prev+critical*se_prev),
            'unstable_flag':n_cases<30 or n_domain<100,'data_cycle':'NHANES 2017-2018',
            'definition':'one-time CKD lab screen AND diagnosed diabetes with onset >=30 (T2D proxy)',
            'source':'https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=2017'}
results=[estimate(d,'All adults','All adults')]
for col in ['age_group','sex','race_ethnicity']:
    for group in d[col].dropna().unique(): results.append(estimate(d,col,group))
result=pd.DataFrame(results)
result.to_csv(ROOT/'population_estimates_2017_2018.csv',index=False)
print(result[['dimension','segment','unweighted_cases','weighted_population_count','prevalence','unstable_flag']].to_string(index=False))

## 5. KG integrity check and provenance exports
No epidemiology values are inserted into biomedical graph nodes in V1. Outputs are separate structured observations, and the original JSON is checked byte-for-byte against the copied baseline.

In [ ]:
source_manifest=pd.DataFrame([{'dataset':name,'url':url,'sha256':hashlib.sha256((ROOT/(name+'.xpt')).read_bytes()).hexdigest(),
                                  'downloaded_file_bytes':(ROOT/(name+'.xpt')).stat().st_size} for name,url in SOURCES.items()])
source_manifest.to_csv(ROOT/'source_manifest.csv',index=False)
report={'baseline_nodes':node_count,'baseline_edges':edge_count,'baseline_sha256':base_hash,
        'copy_unchanged':hashlib.sha256((ROOT/'baseline_kg_COPY.json').read_bytes()).hexdigest()==base_hash,
        'adult_records':len(d),'joint_proxy_evaluable':int(d.joint_proxy.notna().sum()),
        'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'caveat':'Not confirmed CKD, not confirmed T2D, not CKD attributable to T2D, not drug eligibility',
        'no_github_writes':True}
(ROOT/'validation_report.json').write_text(json.dumps(report,indent=2))
assert report['copy_unchanged']
print(json.dumps(report,indent=2))
print('Output files:',*[x.name for x in ROOT.glob('*.csv')],sep='\n - ')

## 6. Export results to Google Drive (optional)
Execute this cell only if you want a durable copy. The target is a **new folder**; it does not touch GitHub. For privacy and size, it copies only result CSVs, provenance and validation report—not raw survey XPT files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
target=Path('/content/drive/MyDrive/Healthcare_KG_Population_Intelligence_V1')
target.mkdir(parents=True,exist_ok=True)
for name in ['population_estimates_2017_2018.csv','source_manifest.csv','validation_report.json','baseline_kg_COPY.json']:
    shutil.copy2(ROOT/name,target/name)
print('Saved to:',target)

## Next development gates
1. Validate national estimates against published NHANES/CDC benchmarks and verify variable coding.
2. Improve diabetes typing (medication/onset sensitivity analyses) and missing-data treatment.
3. Evaluate more recent NHANES releases and harmonized multi-cycle analysis; adjust pooled weights.
4. Only then connect the numeric observation store to the agentic query planner and existing KG.
5. Never interpret the pilot estimates as Kerendia eligibility, incidence, or current-year addressable market.

**References:** [NHANES 2017–2018](https://wwwn.cdc.gov/nchs/nhanes/continuousnhanes/default.aspx?BeginYear=2017); [NHANES survey weighting](https://wwwn.cdc.gov/nchs/nhanes/analyticguidelines.aspx); [2021 CKD-EPI equation](https://www.kidney.org/professionals/kdoqi/gfr_calculator); [CKD definition](https://kdigo.org/guidelines/ckd-evaluation-and-management/).